# 模块二：用 KNN 给水果分类

**30 分钟｜零基础｜AI Coding × 数据实践**

今天交付一个模型：输入重量、直径，输出苹果 / 橙子 / 梨，并解释它参考了哪些邻居。

路径：**规划 → 收集 → 清洗 → 划分 → 建模 → 检查 → 迭代**。AI 帮你写程序，你负责判断数据与结果。

从项目根目录启动 Jupyter，使用 Python 3.10+。按顺序选中单元格，按 **Shift + Enter**。请保留完整项目文件夹，本课需要 `modules/knn/`。无需额外安装机器学习包，只需使用 Python 标准库。

## 0–3 分钟：先定义问题

一行代表**一颗独立水果**。`weight_g`、`diameter_cm` 是特征（输入），`label` 是标签（正确答案），`fruit_id` 只是追踪编号，不参与预测。

KNN 的做法：把新水果和训练样本比较，找距离最近的 K 个邻居，再按票数判断类别。K 是邻居数，不是类别数。训练主要保存样本和缩放参数，预测时只比较新样本和训练样本的距离。

三分类时，即使 K 是奇数也可能平票。本课票数并列时先比较该类别邻居的总距离，再按标签字母顺序决定；距离完全相同的邻居按编号排序，以保证确定性。

先猜猜：165 g、6.8 cm 的水果可能是什么？只凭这两个数字不一定分得清，这正是需要验证的原因。

In [ ]:
from pathlib import Path
from collections import Counter
from html import escape
from IPython.display import HTML, display
from modules.knn.data import LABELS, read_csv, clean_rows, stratified_split
from modules.knn.model import KNNClassifier, select_k, evaluate, majority_baseline

DATA = Path("modules/knn/data/fruits_raw.csv")
names = {"apple": "苹果", "orange": "橙子", "pear": "梨"}
print("环境就绪。当前目录：", Path.cwd())
print("课堂数据存在：", DATA.exists())

## 本 Notebook 使用的项目函数

上面的导入代码已经把本课需要的函数准备好了。它们来自项目中的两个文件：

| 文件 | 已导入对象 | 用途 |
|---|---|---|
| `modules/knn/data.py` | `read_csv` | 读取 CSV 并保留行号 |
| `modules/knn/data.py` | `clean_rows` | 按课堂规则清洗数据并生成审计记录 |
| `modules/knn/data.py` | `stratified_split` | 分层划分训练集、验证集和测试集 |
| `modules/knn/model.py` | `KNNClassifier` | 建立 KNN 分类器、预测和查找邻居 |
| `modules/knn/model.py` | `select_k` | 使用验证集比较候选 K |
| `modules/knn/model.py` | `evaluate` | 计算准确率、混淆矩阵和预测结果 |
| `modules/knn/model.py` | `majority_baseline` | 计算多数类基线 |

学生让 AI 生成代码时，可以告诉 AI：这些对象已经在前面的代码单元中导入，不需要重新实现，也不要修改 `data.py` 或 `model.py`。如果你重启了内核，请先重新运行环境准备单元格。

## 3–8 分钟：规划与采集

真实采集使用 `modules/knn/data/collection_template.csv`，另存一个副本填写。每类目标至少 20 颗不同水果，覆盖不同大小与批次；课堂用模拟数据做演示。

| 字段 | 怎么记录 | 例子 |
|---|---|---|
| fruit_id | 唯一编号，同一颗复测沿用编号 | G1-001 |
| weight_g | 秤归零后的重量，单位 g | 180 |
| diameter_cm | 最宽横截面的直径，单位 cm；不是周长或高度 | 7.2 |
| label | 实物核对后填 apple / orange / pear | apple |

另记日期、器具、商店/批次和标签核对人。只测同一箱水果会限制代表性；同一颗称十次，不能当十颗。

完成真实采集后，将上方 `DATA` 改为你另存的 CSV 路径，再从头运行。

In [ ]:
raw = read_csv(DATA)
print("原始记录数：", len(raw))
print("注意：这是教学模拟数据，包含故意加入的问题记录。")
for record in raw[:3] + raw[-10:]:
    print("CSV 行", record.line, record.values)

## 8–14 分钟：清洗要留下理由

本练习预先限定重量 **50–500 g**、直径 **3–15 cm**。明确标注 kg / mm 才换算；未知单位不能猜；缺失值不能用模型填补。完全重复的同编号记录只保留一条，同编号冲突记录全部隔离。

清洗函数 `clean_rows` 来自 `modules/knn/data.py`，已经在环境准备单元格中导入。学生不需要重新实现。运行后观察一条被标准化的记录和一条被隔离的记录。

In [ ]:
cleaned = clean_rows(raw)
print(f"原始 {len(raw)} 条 → 保留 {len(cleaned.rows)} 颗 → 减少 {len(raw)-len(cleaned.rows)} 条")
print("类别数量：", Counter(row.label for row in cleaned.rows))
print("审计事件数：", len(cleaned.audit), "（一行可能对应多个事件）")
for event in cleaned.audit:
    print(f"第 {event.line} 行 | {event.fruit_id} | {event.code} | {event.message}")

## 14–19 分钟：数据各司其职

训练集约 60%：提供邻居并计算标准化参数；验证集约 20%：选 K；测试集约 20%：最终考试。先按实体去重，再划分。固定随机种子 42，是为了复现。

本部分使用 `stratified_split`，它来自 `modules/knn/data.py`，已经在环境准备单元格中导入。

### Prompt 01：让 AI 生成数据划分代码

> 当前 Notebook 已经导入 `stratified_split`，它来自 `modules.knn.data`。请不要重新实现这个函数。请生成可以直接运行的 Python 代码：
> 1. 使用 `cleaned.rows`；
> 2. 使用随机种子 `42`；
> 3. 将数据划分为训练集、验证集和测试集，并保存为变量 `split`；
> 4. 输出三个集合的样本数和类别分布；
> 5. 检查同一个 `fruit_id` 没有同时出现在多个集合中。
> 只返回 Python 代码。

In [ ]:
# 将 AI 生成的代码粘贴到这里运行


重量和直径的数值尺度不同。若直接计算距离，重量可能因为数值较大而占主导。

**标准化值 =（原值 − 训练集均值）÷ 训练集标准差**。

`KNNClassifier` 来自 `modules/knn/model.py`，它会使用训练集计算标准化参数。验证集、测试集和新水果沿用同一把尺子。零方差特征的缩放因子设为 1。

In [ ]:
demo_model = KNNClassifier.fit(split.train, k=3)
print("训练集均值（g, cm）：", demo_model.scaler.mean)
print("缩放因子：", demo_model.scaler.scale)
print("165 g、6.8 cm 标准化后：", demo_model.scaler.transform(165, 6.8))

## 19–25 分钟：选择 K，然后做一次最终评估

KNN 会对训练数据标准化，计算距离，找到最近的 K 个邻居，再根据投票预测类别。

`KNNClassifier` 和 `select_k` 都来自 `modules/knn/model.py`，已经在环境准备单元格中导入。

### Prompt 02：让 AI 生成 KNN 训练和选 K 代码

> 当前 Notebook 已经导入 `KNNClassifier` 和 `select_k`，它们来自 `modules.knn.model`。请不要重新实现这些对象。请生成可以直接运行的 Python 代码：
> 1. 使用 `split.train` 作为训练集；
> 2. 使用 `split.validation` 比较 K=1、3、5；
> 3. 输出每个 K 的验证集准确率；
> 4. 选择验证集准确率最高的 K；
> 5. 如果准确率相同，选择较小的 K；
> 6. 使用最终选择的 K 创建变量 `classifier`；
> 7. 不要使用 `split.test` 选择 K；
> 8. 不要修改 `modules/knn/model.py`。
> 只返回 Python 代码。

In [ ]:
# 将 AI 生成的代码粘贴到这里运行


下一格是最终考试。流程和 K 确定后再运行。重复运行原封不动的实验可以复现；根据测试结果改参数后，就不能再称原测试集为独立最终考试。

`evaluate` 和 `majority_baseline` 都来自 `modules/knn/model.py`，已经在环境准备单元格中导入。

### Prompt 03：让 AI 生成测试报告代码

> 当前 Notebook 已经导入 `evaluate` 和 `majority_baseline`，它们来自 `modules.knn.model`。请生成可以直接运行的 Python 代码：
> 1. 基于当前的 `classifier` 和 `split.test` 调用 `evaluate()`；
> 2. 输出测试集正确数量和总数量；
> 3. 输出测试集准确率；
> 4. 输出多数类基线；
> 5. 输出中文混淆矩阵；
> 6. 不要重新选择 K，不要重新训练模型。
> 只返回 Python 代码。

In [ ]:
# 将 AI 生成的代码粘贴到这里运行


## 25–27 分钟：查看模型参考了哪些邻居

预测输入应是这三类水果、符合本练习范围；香蕉也可能被硬猜成三类之一。票数占比不是经过校准的概率。

### Prompt 04：让 AI 生成邻居解释代码

> 当前 Notebook 已经导入 `KNNClassifier`，它来自 `modules.knn.model`。请基于当前的 `classifier` 生成可以直接运行的 Python 代码：
> 1. 输入新水果 `new_weight_g = 165.0`；
> 2. 输入新水果 `new_diameter_cm = 6.8`；
> 3. 输出预测类别；
> 4. 输出 K 个邻居；
> 5. 输出每个邻居的排名、编号、类别和标准化距离；
> 6. 类别显示为中文；
> 7. 不要修改模型的预测逻辑。
> 只返回 Python 代码。

In [ ]:
# 将 AI 生成的代码粘贴到这里运行


## 27–28 分钟：看训练样本与新样本的分布

图中只画**训练样本**，黑色十字是新水果，黑圈是被选中的邻居。横纵轴是原始测量值，便于理解；模型算的是**标准化距离**，不能直接按屏幕上的位置解释距离大小。下面的图形代码由课程提供。

In [ ]:
colors = {"apple": "#c94b36", "orange": "#a56800", "pear": "#24744e"}
neighbor_ids = {neighbor.fruit_id for neighbor in neighbors}
points = list(split.train)
x_min = min([row.weight_g for row in points] + [new_weight_g]) - 15
x_max = max([row.weight_g for row in points] + [new_weight_g]) + 15
y_min = min([row.diameter_cm for row in points] + [new_diameter_cm]) - 0.5
y_max = max([row.diameter_cm for row in points] + [new_diameter_cm]) + 0.5
svg = ['<svg viewBox="0 0 720 400" role="img" aria-label="训练水果与新水果的散点图" style="max-width:800px;background:#fff;border:1px solid #ddd">', '<path d="M60 25V340H690" fill="none" stroke="#444" stroke-width="1.2"/>', '<path d="M60 340H690" fill="none" stroke="#444" stroke-width="1.2"/>']
for row in points:
    x = 60 + (row.weight_g-x_min)/(x_max-x_min)*610
    y = 340 - (row.diameter_cm-y_min)/(y_max-y_min)*290
    outline = "#111" if row.fruit_id in neighbor_ids else "white"
    svg.append(f'<circle cx="{x}" cy="{y}" r="6" fill="{colors[row.label]}" stroke="{outline}" stroke-width="2"><title>{escape(row.fruit_id)} {names[row.label]} {row.weight_g}g {row.diameter_cm}cm</title></circle>')
x = 60 + (new_weight_g-x_min)/(x_max-x_min)*610
y = 340 - (new_diameter_cm-y_min)/(y_max-y_min)*290
svg.append(f'<path d="M{x-9} {y}h18 M{x} {y-9}v18" stroke="#111" stroke-width="3"/>')
svg.append(f'<text x="70" y="365" font-size="14">{x_min:.0f} g</text><text x="605" y="365" font-size="14">{x_max:.0f} g</text><text x="10" y="330" font-size="12">{y_min:.1f}</text><text x="10" y="18" font-size="14">直径（cm）</text><text x="290" y="388" font-size="16">重量（g）</text>')
svg.append('</svg>')
display(HTML("".join(svg)))
print("图例：红=苹果，棕黄=橙子，绿=梨；黑圈=邻居，黑十字=新水果。悬停看样本。")

## 28–30 分钟：让 AI 修改一处，自己验证

选择一种挑战：

1. **入门**：让 AI 把清洗统计和分类报告中的英文标签显示成中文，内部 label 保持不变。
2. **进阶**：让 AI 在训练/验证集比较更多合法 K，并解释变化。不要使用测试集挑参数。
3. **解释**：让 AI 改进邻居输出，增加排名和更清晰的格式，不改变预测逻辑。

### Prompt 05：让 AI 做一次小修改

> 请在当前代码基础上完成【我的修改】，保留采集、清洗和数据划分规则。需要使用项目函数时，请使用本 Notebook 前面已经导入的函数，不要重新实现 `data.py` 或 `model.py` 中的函数。先给出可以直接运行的替换代码；不要修改模型核心预测逻辑，也不要使用测试集选择参数。代码尽量简洁，适合当前 Notebook。

In [ ]:
# 将 AI 生成的代码粘贴到这里运行

## 交付：数据决策比高分更重要

- 目标、特征、标签与每行含义。
- 两条清洗记录及理由；三份数据规模。
- 验证集选择的 K、最终测试正确数/总数、准确率与基线。
- 一个新水果的邻居解释及一个使用限制。
- 一次 AI 修改的提示词和运行结果。

不需要写长篇报告，主要提交 Notebook 中的代码和运行结果。

[完整采集指南与教师参考](modules/knn/README.md) · [教师讲稿](KNN_Workshop_Teacher.html) · [返回个人主页 Notebook](AI_Coding_Workshop_Student.ipynb)

概念参考：[最近邻方法](https://scikit-learn.org/stable/modules/neighbors.html)、[数据泄漏](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)。课堂代码使用标准库实现，便于理解原理。